# Titan Arch — Road Segmentation (Step 3b / Phase I, the GPU part)

The only GPU part of Titan Arch, on **free Colab**. Phase I of the spec:
occlusion-robust road segmentation. We train a **baseline U-Net** and an
**advanced Transformer** model (SegFormer / MixVision encoder), with
**occlusion-simulation augmentation** and a **combined Dice + IoU + boundary**
loss, then turn a satellite image into a road mask.

Just give a place name — it fetches the imagery, predicts the mask, and prints
the exact bounding box to paste into the Titan Arch app (which runs the
MST/Disjoint-Set healing + resilience analysis on your laptop).

> Runtime → Change runtime type → **T4 GPU**, then Run all.

In [ ]:
# 1. deps (torch preinstalled on Colab; timm = transformer encoders; rasterio = GeoTIFF)
!pip -q install segmentation-models-pytorch==0.3.4 albumentations==1.4.0 timm==0.9.16 rasterio 2>/dev/null
import torch, segmentation_models_pytorch as smp
print('torch', torch.__version__, '| cuda', torch.cuda.is_available(),
      '|', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU only')

## 2. Road dataset (DeepGlobe, free on Kaggle)
Upload your `kaggle.json` (Kaggle → Account → Create New API Token) when asked.
(SpaceNet / OpenSatMap also work — point `imgs` at any `*_sat.jpg` + `*_mask.png`.)

In [ ]:
import os, glob
from google.colab import files
if not os.path.exists('/root/.kaggle/kaggle.json'):
    print('Upload kaggle.json:'); files.upload()
    !mkdir -p /root/.kaggle && cp kaggle.json /root/.kaggle/ && chmod 600 /root/.kaggle/kaggle.json
!pip -q install kaggle 2>/dev/null
!kaggle datasets download -d balraj98/deepglobe-road-extraction-dataset -p /content/dg --unzip 2>/dev/null
imgs = sorted(glob.glob('/content/dg/train/*_sat.jpg'))
print('training tiles found:', len(imgs))

In [ ]:
# 3. data + OCCLUSION-SIMULATION augmentation (Phase I: simulate tree/shadow gaps)
import numpy as np, cv2, torch
from torch.utils.data import Dataset, DataLoader
import albumentations as A
SIZE = 512
# CoarseDropout punches blank patches into the IMAGE only (mask untouched) -> the
# model must infer road continuity under occlusion, exactly the spec's goal.
train_aug = A.Compose([
    A.RandomCrop(SIZE, SIZE), A.HorizontalFlip(), A.VerticalFlip(), A.RandomRotate90(),
    A.RandomBrightnessContrast(0.2, 0.2),
    A.CoarseDropout(max_holes=10, max_height=48, max_width=48, min_holes=3,
                    fill_value=0, mask_fill_value=None, p=0.7),   # simulate occlusion
])
val_aug = A.Compose([A.CenterCrop(SIZE, SIZE)])
class RoadDS(Dataset):
    def __init__(self, paths, aug): self.paths, self.aug = paths, aug
    def __len__(self): return len(self.paths)
    def __getitem__(self, i):
        sp = self.paths[i]; mp = sp.replace('_sat.jpg', '_mask.png')
        img = cv2.cvtColor(cv2.imread(sp), cv2.COLOR_BGR2RGB); msk = cv2.imread(mp, 0)
        a = self.aug(image=img, mask=msk); img, msk = a['image'], a['mask']
        img = (img/255.).astype('float32').transpose(2,0,1)
        msk = (msk > 127).astype('float32')[None]
        return torch.from_numpy(img), torch.from_numpy(msk)
split = int(len(imgs)*0.95)
tr = DataLoader(RoadDS(imgs[:split], train_aug), batch_size=4, shuffle=True, num_workers=2)
va = DataLoader(RoadDS(imgs[split:], val_aug), batch_size=4, num_workers=2)
print('train batches', len(tr), '| val batches', len(va))

In [ ]:
# 4. model + COMBINED LOSS (Phase I: baseline U-Net OR advanced Transformer)
dev = 'cuda' if torch.cuda.is_available() else 'cpu'
#   BACKBONE = 'resnet34'  -> baseline U-Net + ResNet (CNN)
#   BACKBONE = 'mit_b0'    -> ADVANCED transformer encoder (SegFormer / MixVision,
#                            attention + long-range context, the spec's Phase I goal)
BACKBONE = 'mit_b0'
model = smp.Unet(BACKBONE, encoder_weights='imagenet', in_channels=3, classes=1).to(dev)

# combined Dice + IoU(Jaccard) + BCE + a light boundary-aware term
loss_dice = smp.losses.DiceLoss('binary')
loss_jac  = smp.losses.JaccardLoss('binary')          # IoU loss
loss_bce  = torch.nn.BCEWithLogitsLoss()
def boundary_loss(logits, y):
    # emphasise road edges: BCE weighted by how close a pixel is to a boundary
    p = torch.sigmoid(logits)
    k = torch.ones(1, 1, 3, 3, device=y.device)
    edge = (torch.nn.functional.conv2d(y, k, padding=1) % 9 != 0).float()  # mixed nbhd
    w = 1.0 + 4.0 * edge
    return (w * (p - y).abs()).mean()
def criterion(logits, y):
    return loss_dice(logits, y) + loss_jac(logits, y) + loss_bce(logits, y) \
        + 0.5 * boundary_loss(logits, y)
opt = torch.optim.Adam(model.parameters(), lr=3e-4)
def iou(logits, y, t=0.5):
    p = (logits.sigmoid() > t).float()
    inter = (p*y).sum((1,2,3)); union = ((p+y)>0).float().sum((1,2,3))
    return ((inter+1e-6)/(union+1e-6)).mean().item()
print('model:', BACKBONE, '| params (M):', round(sum(p.numel() for p in model.parameters())/1e6, 1))

In [ ]:
# 5. train (raise EPOCHS for more accuracy)
EPOCHS = 6
for ep in range(EPOCHS):
    model.train(); tl = 0
    for x, y in tr:
        x, y = x.to(dev), y.to(dev); opt.zero_grad()
        loss = criterion(model(x), y)
        loss.backward(); opt.step(); tl += loss.item()
    model.eval(); vi = 0
    with torch.no_grad():
        for x, y in va: x, y = x.to(dev), y.to(dev); vi += iou(model(x), y)
    print(f'epoch {ep+1}/{EPOCHS}  loss={tl/len(tr):.3f}  val_IoU={vi/max(1,len(va)):.3f}')
torch.save(model.state_dict(), '/content/titanarch_road_unet.pt')
print('saved weights')

## 6. Get a satellite image → road mask

Two ways to supply imagery:
- **Auto-fetch** by place name (below), or
- **A real GeoTIFF** (Resourcesat **LISS-IV**, **Cartosat-3**, **Sentinel-2**): set
  `GEOTIFF = '/content/your_tile.tif'` and we read the pixels *and* the bounds with
  rasterio — no coordinates to type.

Either way you get `road_mask.png` **and** a georeferenced `road_mask.tif`; upload
the `.tif` to Titan Arch and the bounds are read automatically.

In [ ]:
# --- imagery source: a real GeoTIFF, or auto-fetched Esri tiles (no key) ---
import math, io, requests, numpy as np
from PIL import Image
UA = {'User-Agent': 'TitanArch/0.1'}
ESRI = 'https://server.arcgisonline.com/ArcGIS/rest/services/World_Imagery/MapServer/tile/{z}/{y}/{x}'
def _llt(lat, lon, z):
    n = 2**z; return (lon+180)/360*n, (1-math.asinh(math.tan(math.radians(lat)))/math.pi)/2*n
def _tll(x, y, z):
    n = 2**z; return math.degrees(math.atan(math.sinh(math.pi*(1-2*y/n)))), x/n*360-180
def fetch_imagery(bbox, zoom=18, max_tiles=80):
    s, w, n_, e = bbox
    x0f, y0f = _llt(n_, w, zoom); x1f, y1f = _llt(s, e, zoom)
    x0, x1 = int(x0f//1), int(x1f//1); y0, y1 = int(y0f//1), int(y1f//1)
    nx, ny = x1-x0+1, y1-y0+1; assert nx*ny <= max_tiles, f'{nx*ny} tiles; lower zoom'
    cv = Image.new('RGB', (nx*256, ny*256))
    for ix in range(x0, x1+1):
        for iy in range(y0, y1+1):
            r = requests.get(ESRI.format(z=zoom, x=ix, y=iy), headers=UA, timeout=30); r.raise_for_status()
            cv.paste(Image.open(io.BytesIO(r.content)).convert('RGB'), ((ix-x0)*256, (iy-y0)*256))
    north, _ = _tll(x0, y0, zoom); south, _ = _tll(x1+1, y1+1, zoom)
    _, west = _tll(x0, y0, zoom); _, east = _tll(x1+1, y1+1, zoom)
    return np.asarray(cv), (south, west, north, east)
def geocode(place, half_m=900):
    j = requests.get('https://nominatim.openstreetmap.org/search',
                     params={'q': place, 'format': 'json', 'limit': 1}, headers=UA, timeout=30).json()
    lat, lon = float(j[0]['lat']), float(j[0]['lon'])
    dlat = half_m/111320; dlon = half_m/(111320*max(0.2, math.cos(math.radians(lat))))
    return (lat-dlat, lon-dlon, lat+dlat, lon+dlon)

GEOTIFF = None                      # <-- set to a LISS-IV/Cartosat/Sentinel-2 .tif to use it
PLACE   = 'Indiranagar, Bengaluru'  # else auto-fetch this place
ZOOM    = 18

if GEOTIFF:
    import rasterio
    from rasterio.warp import transform_bounds
    with rasterio.open(GEOTIFF) as ds:
        b = ds.read()[:3]                                 # first 3 bands -> RGB
        sat = np.transpose(b, (1, 2, 0))
        if sat.max() > 255: sat = (sat / sat.max() * 255)
        sat = sat.astype('uint8')
        l, bo, r_, t = ds.bounds
        w, s, e, n_ = transform_bounds(ds.crs, 'EPSG:4326', l, bo, r_, t) \
            if ds.crs and ds.crs.to_epsg() != 4326 else (l, bo, r_, t)
        exact_bbox = (s, w, n_, e)
    print('GeoTIFF', sat.shape, '| bbox', tuple(round(v, 5) for v in exact_bbox))
else:
    bbox = geocode(PLACE, half_m=900)
    sat, exact_bbox = fetch_imagery(bbox, zoom=ZOOM)
    print('imagery', sat.shape, '| exact bbox', tuple(round(v, 5) for v in exact_bbox))

In [ ]:
# predict the road mask (tiled + stitched) and show it next to the imagery
import matplotlib.pyplot as plt
def predict_mask(img, tile=512, overlap=64, thr=0.5):
    H, W = img.shape[:2]; prob = np.zeros((H, W), 'float32'); cnt = np.zeros((H, W), 'float32')
    step = tile-overlap; model.eval()
    with torch.no_grad():
        for y0 in range(0, max(1, H-1), step):
            for x0 in range(0, max(1, W-1), step):
                y0c, x0c = min(y0, H-tile) if H>=tile else 0, min(x0, W-tile) if W>=tile else 0
                y0c, x0c = max(0, y0c), max(0, x0c)
                patch = img[y0c:y0c+tile, x0c:x0c+tile]; ph, pw = patch.shape[:2]
                t = np.zeros((tile, tile, 3), 'uint8'); t[:ph, :pw] = patch
                x = torch.from_numpy((t/255.).astype('float32').transpose(2,0,1))[None].to(dev)
                p = model(x).sigmoid()[0,0].cpu().numpy()[:ph, :pw]
                prob[y0c:y0c+ph, x0c:x0c+pw] += p; cnt[y0c:y0c+ph, x0c:x0c+pw] += 1
    return ((prob/np.maximum(cnt,1)) > thr).astype('uint8')*255
mask = predict_mask(sat)
Image.fromarray(mask).save('road_mask.png')
fig, ax = plt.subplots(1, 2, figsize=(15, 7))
ax[0].imshow(sat); ax[0].set_title('satellite'); ax[0].axis('off')
ax[1].imshow(mask, cmap='gray'); ax[1].set_title(f'predicted roads ({BACKBONE})'); ax[1].axis('off'); plt.show()

In [ ]:
# 7. export the mask (georeferenced GeoTIFF + PNG) and download
from google.colab import files
s, w, n_, e = exact_bbox

# georeferenced GeoTIFF -> Titan Arch reads the bounds automatically (no typing)
try:
    import rasterio
    from rasterio.transform import from_bounds
    tr = from_bounds(w, s, e, n_, mask.shape[1], mask.shape[0])
    with rasterio.open('road_mask.tif', 'w', driver='GTiff', height=mask.shape[0],
                       width=mask.shape[1], count=1, dtype='uint8',
                       crs='EPSG:4326', transform=tr) as ds:
        ds.write(mask.astype('uint8'), 1)
    print('Upload road_mask.tif to Titan Arch -> "upload your own road mask" (bounds auto-read).')
    files.download('road_mask.tif')
except Exception as ex:
    print('GeoTIFF export skipped:', ex)
    print(f'Paste bbox manually:  South={s:.6f} West={w:.6f} North={n_:.6f} East={e:.6f}')

files.download('road_mask.png')